# 05 Supervised linear probe

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/claude/awesome-archimedes-kvh4c7/notebooks/05_supervised_probe.ipynb)

No GPU is needed. Run `03_zero_shot.ipynb` first: this notebook reads the baseline's embeddings and predictions.

**What it does.** Trains a logistic regression (a linear probe) on stored embeddings and reports how well it does on photos it did not train on. This is the "supervised probe" of the algorithm ideas (idea 4), rewritten so that its numbers can be trusted: labels join to embeddings on `(image_name, coral_id)`, cross-validation is grouped by photo, class weights are balanced, and **only held-out predictions are reported**, never accuracy on the training data.

**Where the labels come from.** The probe never makes its own labels, and zero-shot predictions are never used as labels. Today the only labels are the expert test set (`assaf_expert_test_set.csv`, 77 rows). This notebook uses the **51 rows on landscape photos whose genus is one of the 20**. It leaves out the soft corals, the `unidentifiable` and `MISSING_LABEL` rows, and the 10 rows on portrait photos: Assaf labeled those on crops the legacy code cut from the wrong place, so his label does not describe the crop the probe sees.

**Read the result as a template check, not a finding.** 51 labels across up to 20 genera, several with one or two examples, is far too little to mean anything: the held-out accuracy has a wide interval and a few crops move it by points. The notebook exists to show that the pipeline runs end to end. The real use comes with the larger, stratified, blind expert set (about 300 crops, `docs/algorithm_ideas.md` idea 1.1): put it in a CSV or parquet with `image_name`, `coral_id` and a `genus` column, and set `LABELS_CSV` below.

**Safe to run and re-run.** It reads the shared Drive and only creates new files: each run writes a new folder `<run>/reports/probe_<name>_<date>/`.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[probe,viz]` | 2-4 min | GitHub | nothing on Drive |
| Config | loads the run config, checks the inputs | seconds | the run folder | nothing |
| Labels | the expert set's usable rows, with a count of every row left out and why | seconds | `assaf_expert_test_set.csv`, the manifest | nothing |
| Probe | grouped cross-validation, held-out report, zero-shot on the same crops | seconds for 51 labels; about a minute for a few hundred | `features/<FEATURES>.npz`, the baseline predictions | `<run>/reports/probe_<name>_<date>/` (a new folder) |
| Tables | per-genus precision and recall, the top confusions, the folds, the confusion matrix | seconds | the report | nothing |

**What to paste back to the next session:** the printed label summary and held-out accuracy lines.

In [ ]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "claude/awesome-archimedes-kvh4c7"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "probe,viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

In [ ]:
import json
import logging
from pathlib import Path

import pandas as pd

from coralid.config import load_config
from coralid.manifest import read_manifest

# ================================ SETTINGS ================================
LABELS_CSV = None  # None: the expert set's landscape rows. Later: a labels table (CSV or parquet
# with image_name, coral_id and a genus column), a path under DATA_ROOT or absolute
LABEL_COLUMN = "genus"  # the genus column of LABELS_CSV
FEATURES = "finetuned-bioclip__masked"  # the embeddings to train on: a file stem in <run>/features/
N_SPLITS = 5  # cross-validation folds (the folds hold whole photos)
# ==========================================================================

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
cfg.probe.features, cfg.probe.n_splits = FEATURES, N_SPLITS
EXPERT_CSV = cfg.run_dir / "assaf_expert_test_set.csv"
for label, path in {
    "manifest": cfg.manifest_path,
    "features": cfg.features_dir / f"{FEATURES}.npz",
    "baseline": cfg.predictions_dir / "finetuned-bioclip__taxonomy_v1__hard20__all.parquet",
    "labels": EXPERT_CSV if LABELS_CSV is None else cfg.data_root / LABELS_CSV,
}.items():
    print(f"  {label:9} {'ok' if path.exists() else 'MISSING':8} {path}")
manifest = read_manifest(cfg.manifest_path)

## The labels

Every expert row is accounted for below: it is either a label or counted under the reason it was left out (outside the 20 genera, unidentifiable, missing, on a portrait photo, or a photo that is not in the manifest).

In [ ]:
from coralid.probe import load_labels
from coralid.probe_labels import expert_labels

if LABELS_CSV is None:
    labels, label_summary = expert_labels(EXPERT_CSV, manifest, cfg.probe.label_set)
    cfg.probe.label_column = "genus"
    print("Expert set -> labels (landscape rows in the label set only):")
    print(json.dumps(label_summary, indent=1))
    report_name = f"{FEATURES}__expert_landscape"
else:
    labels = load_labels(cfg.data_root / LABELS_CSV)
    cfg.probe.label_column = LABEL_COLUMN
    print(f"{len(labels)} labels from {LABELS_CSV}")
    report_name = f"{FEATURES}__{Path(LABELS_CSV).stem}"
display(labels[cfg.probe.label_column].value_counts().rename("labels").to_frame())

## The probe

`train_probe` joins the labels to the embeddings by key, cross-validates by photo, and saves a report with the held-out predictions of every labeled crop. The comparison line scores the baseline's zero-shot predictions on the same crops, so the probe has something to beat. With so few labels the probe will often not beat it: that is the expected result here.

In [ ]:
from coralid.jobs import train_probe
from coralid.reports import probe_report_lines

report, data, folder = train_probe(cfg, labels, name=report_name)
for line in probe_report_lines(report, data.dropped):
    print(line)
print("  report (a new folder):", folder)

In [ ]:
from coralid.viz import confusion_heatmap

print("Held-out precision and recall per genus (genera with no labels are not shown):")
display(report.evaluation.per_genus.round(3))
print("Top confusions (true -> predicted):")
display(report.evaluation.top_confusions)
print("The folds:")
display(pd.DataFrame(report.folds).round(3))
display(confusion_heatmap(report.evaluation.confusion, title="Held-out confusion (true in rows)"))

## Next

- Paste the label summary and the held-out accuracy lines into the next session.
- When the larger expert set exists, set `LABELS_CSV` to it and run again: the per-genus table and the comparison with zero-shot become meaningful then.
- The report folder holds `held_out.csv` (the label and the probe's prediction for every crop), `per_genus.csv`, `confusion.csv`, `folds.csv` and `summary.json`.